### **Initialize OpenAI API**

In [ ]:
!pip install openai
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
!pip install duckduckgo-search
!pip install -U pydantic
!pip install crewai_tools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.0/375.0 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 4.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of embedchain to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of embedchain to determine which ve

In [ ]:
from crewai import Agent, Task, Crew, process
import langchain
from google.colab import userdata

In [ ]:
import os
from google.colab import userdata

# Retrieve the API key from userdata
openai_api_key = userdata.get('ccipynb')

# Check if the API key is retrieved correctly
if openai_api_key:
    print("API Key retrieved successfully.")
else:
    print("API Key retrieval failed.")

# Set the environment variable
os.environ['OPENAI_API_KEY'] = openai_api_key

# Optionally, you can also set the model name
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

API Key retrieved successfully.


In [ ]:
import openai

from openai import OpenAI

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": "Hello!"}]
)
print(response)

ChatCompletion(id='chatcmpl-A99sCz0Gu3Pdx0ubv9uEtzIKtbHzu', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I assist you today?', refusal=None, role='assistant', function_call=None, tool_calls=None))], created=1726746268, model='gpt-3.5-turbo-0125', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=9, prompt_tokens=9, total_tokens=18, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=0)))


In [ ]:
from crewai import Agent, Task, Crew, Process
from langchain_community.tools import DuckDuckGoSearchRun
import requests
from bs4 import BeautifulSoup
import time
from requests.exceptions import RequestException
from langchain_community.document_loaders import AsyncChromiumLoader
from langchain_community.document_transformers import BeautifulSoupTransformer

In [ ]:
# To test whether the ChatAPI is set
response_content = response.choices[0].message.content
print(response_content)

Hello! How can I assist you today?


### **Creating** **Tools**

In [ ]:
# Scraping Tools
search_tool = DuckDuckGoSearchRun()
def web_scrape_tool(url, max_retries=3, initial_timeout=30):
    def exponential_backoff(attempt):
        return min(60, initial_timeout * (2 ** attempt))

    for attempt in range(max_retries):
        try:
            timeout = exponential_backoff(attempt)
            response = requests.get(url, timeout=timeout)
            response.raise_for_status()  # Raises an HTTPError for bad responses
            soup = BeautifulSoup(response.content, 'html.parser')
            return soup.get_text()
        except RequestException as e:
            if attempt == max_retries - 1:
                return f"Error scraping the website after {max_retries} attempts: {str(e)}"
            print(f"Attempt {attempt + 1} failed. Retrying in {timeout} seconds...")
            time.sleep(timeout)

def web_scraper(url): # compared 4 methods (and FirecrawlScrapeWebsiteTool and ScrapeWebsiteTool(cant handle antiscraping) provided by crewAI), this one most recommended
    loader = AsyncChromiumLoader(url)
    html = loader.load()
    # Transform
    bs_transformer = BeautifulSoupTransformer()
    docs_transformed = bs_transformer.transform_documents(html, tags_to_extract=["span"])
    return docs_transformed

In [ ]:
# File Read Tool
from crewai_tools import tool

#@tool("read_file")
def read_file(file_path: str) -> str:
    """
    Reads the content of a file at the given file path.

    Args:
        file_path (str): The path to the file to be read.

    Returns:
        str: The content of the file if successfully read, or an error message if an exception occurs.
    """
    try:
        with open(file_path, 'r') as file:
            content = file.read()
        return content
    except FileNotFoundError:
        return f"Error: File not found at {file_path}"
    except IOError:
        return f"Error: Unable to read file at {file_path}"
    except Exception as e:
        return f"An unexpected error occurred: {str(e)}"


read_cv_using_read_file = read_file("/content/drive/MyDrive/CVs/CV0508.md")
print(read_cv_using_read_file)

Error: File not found at /content/drive/MyDrive/CVs/CV0508.md


### **Agents**

In [ ]:
PostExtracter = Agent(
    role="Job posting information extractor",
    goal="Extract key information from job postings",
    backstory="You are a key member of a career coaching team, dedicated to helping your client secure the job posted at URL: {PostingURL}."
              "You are an expert at analyzing job postings and extracting crucial information for job seekers."
              "As the Job Posting Analyzer, your expertise is crucial in extracting and interpreting vital information from job listings.",
    allow_delegation=False,
    verbose=True,
    Tools = [web_scraper]
)

In [ ]:
#TODO: ClientKnower, by asking client questions, analyse the uploaded resume/record/achievements files,
#linkedinpage and information provided by the client, the CKer have really good understanding of client's experience and ability.

ClientKnower = Agent(
    role="Personal Profiler for Engineers",
    goal="Develop a comprehensive understanding of the client's experience, skills, and qualifications using her resume {resumeContent}.",
    backstory="Equipped with analytical prowess, you dissect "
              "and synthesize information "
              "from diverse sources to craft comprehensive "
              "personal and professional profiles, laying the "
              "groundwork for personalized resume enhancements."
              ,
    allow_delegation=False,
    verbose=True,
    Tools=[read_file]
)

In [ ]:
#TODO: later add tool googlemap(api?) to calculate distance of the company
CompanyInvestigater = Agent(
    role="Company packground Investigater",
    goal="Investigate the company that posts the job position at URL:{PostingURL}",
    backstory="You are a vital member of a career coaching team, focused on helping your client secure the job at URL: {PostingURL}."
              "As the CompanyInvestigator, your expertise lies investigating the company that post the job postion and summarize information about the company."
              "Your role is crucial in providing comprehensive insights about the company, which aids in tailoring the client's application and interview preparation."
              "This information is essential for aligning the client's profile with the company's culture and requirements,"
              "optimizing their application strategy, and preparing them for informed discussions during interviews."
              "Your insights contribute significantly to the team's overall goal of positioning the client as an ideal candidate for the position.",
    allow_delegation=False,
	verbose=True
)

In [ ]:
ResumeImprover = Agent(
    role="Resume Improver",
    goal="Refine, rewrite, extend, modify or restructure your client's resume to improve her chance to get the job.",
    backstory="You are working in a carrier coach team to help you client to get the job at URL:{PostingURL}."
              "The task of your team includes but not limited to: help client modify, optimize her resume,"
              "find out how does your client's ability and experience align with the requirement of the job position"
              "and provide suggestions to answer question in the following Interviews. "
              "You imporve the client's resume based on the work of ClientKnower, CompanyInvestigater and PostExtracter"
              "who provide relevant infomations about the company background and the job position, and also knowledge about client's experience and ability."
              ,
    allow_delegation=False,
	verbose=True
)

In [ ]:
ResumeTranslator = Agent(
    role="Resume Translator",
    goal="Translate client's resume from English to German",
    backstory="You are a well experienced career coach who have seen a lot of resume in both english and german language"
              "You translte the client's resume from English to German with top accuracy."
              "You audit and check typo after trasnlation to make sure it's consitent and have no grammar mistake."
              ,
    allow_delegation=False,
	verbose=True
)

In [ ]:
CoverLetterWriter = Agent(
    role="Cover letter writer",
    goal="Based on your client's background and the job posting, write her a one page cover letter to highlight her relevant experience, interest and motivation.",
    backstory="You are working in a carrier coach team to help you client to get the job at URL:{PostingURL}."
              "The task of your team includes but not limited to: help client modify, optimize her resume,"
              "find out how does your client's ability and experience align with the requirement of the job position"
              "and provide suggestions to answer question in the following Interviews. "
              "You generate the cover letter based on the work of ClientKnower, CompanyInvestigater and PostExtracter"
              "who provide relevant infomations about the company background and the job position, and also knowledge about client's experience and ability."
              ,
    allow_delegation=False,
    Tools=[web_scraper],
	verbose=True
)

In [ ]:
CLFactChecker = Agent(
    role="Cover letter Fact Checker",
    goal="Make sure the truthfullness of generated cover letter, modify it if needed",
    backstory="You are working in a carrier coach team to help you client to get the job at URL:{PostingURL}."
              "The task of your team includes but not limited to: help client modify, optimize her resume,"
              "find out how does your client's ability and experience align with the requirement of the job position"
              "and provide suggestions to answer question in the following Interviews. "
              "You check the cover letter based on the work of CoverLetterWriter, who will generate a draft of cover letter that may contain faked facts about your client."
              "You take the responsibility to check it in detail about every fact, to see whether it align with her resume: {resumeContent}"
              "Once you find faked fact in the cover letter, you either remove it or modify it with her real relevant experience."
              ,
    allow_delegation=False,
    Tools=[web_scraper],
	verbose=True
)

### **Tasks**

In [ ]:
from pydantic import BaseModel
# Define a pydantic model for Job Descriptions
class JobDescription(BaseModel):
    Job_Title: str
    Company_Name: str
    Job_Description: str
    Key_Responsibilities: str
    Qualifications_and_Requirements: str
    Location: str
    Benefits_Salary: str
    Company_Overview: str

AboutJob = Task(
    description=(
        "Analyze the job posting at {PostingURL} and extract the following information:\n"
        "If any information is not found, state 'Information not available'."
        "Extract Company name and save it in {comp_name}"
        "Use the original language of the job posting."
        "Use markdown formatting for better readability."
        "Use the original language of the job posting."
        "Read through the job posting carefully, and list all possible locations for the job if there are multiple" #still can't list all cities
        "For the requiredments, qualifications and responsibilities, use original text in that url."
    ),
    expected_output=(
        "A structured summary of the job posting containing the following information:\n"
        "1. Job Title\n"
        "2. Company Name\n"
        "3. Job Description (brief summary and a comprehensive summary)\n"
        "4. Key Responsibilities\n"
        "5. Qualifications and Requirements\n"
        "6. Location\n "
        "7. Benefits and Salary (if available)\n"
        "8. Company Overview (brief)\n"
        "If any information is not found, state 'Information not available'."

    ),
    output_json=JobDescription,
    output_file="job_description.json", # but where to save the file?
    #comp_name=output_json.Company_Name,
    #job_position=output_json.Job_Title,
    #ouput_file=f"job_description_{comp_name}_{job_position}",
    agent=PostExtracter,
    #tools=[web_scrape_tool, search_tool], # Scrap the original url and search online about this job
    Tools=[web_scrape_tool], # Only use scraper works better, web_scrape_tool lead to untrue info
    #async_execution=True, # Need: create a folder to save everything about this comp, including: job description, company summary and refined resume
)

In [ ]:
# TODO: may need CodeInterpreterTool to use python to read info from pdf
#and her personal statement {personal_s}
AboutClient = Task(
    description=(
        "Compile a detailed personal and professional profile "
        "based on the client's resume. Utilize tools to extract and "
        "synthesize information from her resume from {resumeContent} and her {Personal_Statement}."
        "Use the original language of the job posting."
        "Use markdown formatting for better readability."
    ),
    expected_output=("A comprehensive profile document based on the client's resume that includes her skills, "
        "project experiences, educations, contributions, interests. in markdown format")
        ,
    agent=ClientKnower,
    #Tools=[web_scrape_tool], # This tool not good at reading pdf contents in a url# Need tools to read resume and record files (fileReader), scrape linkedin url (web_scraper)
    Tools=[read_file],
    output_file="client_profile.md"
)

In [ ]:
# TODO: still using duckduckgosearch, can't get correct company name. Serper not really working, no usage
from crewai_tools import SerperDevTool
serper_api_key = userdata.get('serper')
os.environ["SERPER_API_KEY"] = serper_api_key
serper_tool = SerperDevTool()
class CompInfo (BaseModel):
    serper_result: str
    company_name: str
    company_core_business: str
    company_location: str
    company_kununu_score: str
    employee_reviews: str
    number_employee: str

AboutCompany = Task(
    description=(
        "Compile a structured summary of the company {comp_name}, including key information such as:"
        "The company's core business, products, or services"
        "The company's location"
        "The company's kununu score, reputation overall employee reviews"
        "Utilize tools to search this company, and visit the first 5 websites of serper result, output the serper result, run web scraping on those websites"
        "Then summarize information collected from those websites and output them in markdown format"
        "Company size, structure and number of employees"
        "If you can find some of the key informations, don't fake it just say you can't find it"
        "Use the original language of the job posting."
    ),
    expected_output="A clear and well structured summary of the company, including key information such as:"
                    "The company's core business, products, or services"
                    "The company's position. If there are multiple list all of them"
                    "The company's kununu score, reputation overall employee reviews"
                    "Financial stability and growth" #may need integrate external tool to assess this
                    "Company size, structure and number of employees"
        ,
    agent=CompanyInvestigater,
    Tools=[serper_tool, web_scraper],
    context=[AboutJob], # Use output of AboutJob as input for this task
    output_json=CompInfo,
    output_file="company_profile.json",
)

In [ ]:
RefineResume = Task(
    description=(
        "Refine, rewrite, extend, modify or restructure your client's resume {resumeContent} to improve her chance to get the job."
        "Make use of information provided by ClientKnower to tailor her resume for the job, highlighting her skills and experiences most relevant to that position."
        "If the client's experience or ability align with job requirement, use keywords from the job posting to help her resume pass applicant tracking systems (ATS)."
        "Here are the principles of how you should improve her resume:"
              "1. Make sure the information of client that align with the job's requirement is prioritized in the resume."
              "2. Make use of information provided by ClientKnower to make the resume comprehensive"
              "3. Use action verbs and concrete examples rather than vague statements"
              "4. Use a clean, professional font and formatting that is easy to read"
              "5. Proofread carefully for any errors or typos"
              "6. Do not fake any information that is not true about the client"
              "7. Use the original language of the job posting."
              "8. Use markdown formatting for better readability."
    ),
    expected_output=("A comprehensive and well structured resume document in markdown format"
                    "using the original resume overall structure and text style."
                    "Complemented with any additional background information that are align with the position requirement."
                    "With the most relevant expereience or ability prioritized in each section.")
        ,
    agent=ResumeImprover,
    context=[AboutClient, AboutJob], # Need tool: pdfGenerator, save the refined resume in the company's folder
    output_file="refined_resume.md",
    # output_file=f"RefCV4_{comp_name}_{job_position}"
)

In [ ]:
TranslateResume = Task(
    description=(
        "Translate your client's resume {resumeContent} from English to German"
    ),
    expected_output=("A comprehensive and well structured translated resume document in Microsoft Word (.docx) format and in german"
                    "The output .docx file should use the same resume structure, font and layout as the original .docx file and text style."
                    "The translted german resume should be typo and grammar mistake free")
        ,
    agent=ResumeTranslator,
    context=[AboutClient], # Need tool: pdfGenerator, save the refined resume in the company's folder
    output_file="resume_german.docx",
    Tools=[read_file],
    # output_file=f"RefCV4_{comp_name}_{job_position}"
)

In [ ]:
WriteCL = Task(
    description=(
        "Write a compelling cover letter for your client that effectively showcases their qualifications for the job position."
        "Follow these guidelines:"
        "1. Tailor the content to the specific job and company, using information from AboutJob, AboutClient, and her {Motivation} for the job."
        "2. Highlight the client's most relevant skills and experiences that align with the job requirements."
        "3. Use a professional yet approachable tone, avoiding overly complex language."
        "4. Incorporate keywords from the job posting to improve ATS compatibility."
        "5. Structure the letter with a clear introduction, body paragraphs, and conclusion."
        "6. Keep the content concise and limit it to one page."
        "7. Use the original language of the job posting."
        "8. Proofread carefully for any errors or typos."
        "9. Use markdown formatting for better readability."
        "10. Provide a title"
        "Most importantly, only use information from her resume, do not fake any experience that is not in her resume."
        "Include all information from her {Motivation}"
        "do not use too much word to praise the company, maximal one sentense."
        "The overall tone should be modest, honest and true"
        "Do not fake any information that's not true about the client"
    ),
    expected_output=(
        "A well-crafted, one-page cover letter in markdown format that:"
        "- Is tailored to the specific job and company"
        "- Highlights the client's most relevant qualifications"
        "- Incoporate her motivation for the job"
        "- Uses a professional but easily understandable tone"
        "- Incorporates relevant keywords from the job posting"
        "- Is free of errors and typos"
    ),
    agent=CoverLetterWriter,
    context=[AboutClient, AboutJob],
    output_file="cover_letter.md",
)

In [ ]:
CLFactCheck = Task(
    description=(
        "Review the generated cover letter, check whether its stated facts are true as writen in her resume: {resumeContent}"
        "Remove all parts that are not true about her, keep only the true parts."
        "If the cover letter become shorter than half page after removal, pick some relevant experience from her resume {resumeContent} to complete the cover letter. "
        "Most importantly, only use information from her resume: {resumeContent}, do not fake any experience that is not in her resume."
        "Make sure all information from her {Motivation} are included"
    ),
    expected_output=(
        "A well-crafted, half to one page cover letter in markdown format that:"
        "- Is tailored to the specific job and company, while maintaining the truthfulness of her experience stated in the cover letter"
        "- Highlights the client's most relevant qualifications"
        "- Incoporate her motivation: {Motivation} for the job"
        "- Uses a professional but easily understandable tone"
        "- Incorporates relevant keywords from the job posting"
        "- Is free of errors and typos"
    ),
    agent=CLFactChecker,
    context=[AboutClient, AboutJob, WriteCL],
    output_file="cover_letter_checked.md",
)

### **Kick Off**

In [ ]:
crew = Crew(
    agents=[PostExtracter, ClientKnower, CoverLetterWriter],
    tasks=[AboutJob, AboutClient, WriteCL],
    verbose=2
)

result = crew.kickoff(inputs={
    "Motivation":"She has both experience for ML in her study and work. For ML, she assited in transpile project codes using LLM frameworks and also used LLM to develop her personalized tool to help her job applications. In this process, she accquired a lot knowledge on varies LLM models and fine tuning techniques, in her study at university she was also involved in many data science projects. In general what motivate her most is to use novel technoque to build practical applicaiton, that can help people indeed. She believe technologies should serve for people ultimately. Therefore, she is very interested in the reasearch work that can combine social science and ML. ",
    "PostingURL":"https://jobs.dfki.de/ausschreibung/researcher-spatiotemporal-modelling-and-event-data-analysis-567802.html",
    "resumeContent":read_cv_using_read_file,
    "comp_name":"default",
    "Personal_Statement":"She likes to do varies of sports in her free time, including bouldering, running and powerlifting."})

 [DEBUG]: == Working Agent: Job posting information extractor
 [INFO]: == Starting Task: Analyze the job posting at https://jobs.dfki.de/ausschreibung/researcher-spatiotemporal-modelling-and-event-data-analysis-567802.html and extract the following information:
If any information is not found, state 'Information not available'.Extract Company name and save it in defaultUse the original language of the job posting.Use markdown formatting for better readability.Use the original language of the job posting.Read through the job posting carefully, and list all possible locations for the job if there are multipleFor the requiredments, qualifications and responsibilities, use original text in that url.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer:
1. Job Title
Researcher - Spatiotemporal Modelling and Event Data Analysis

2. Company Name
DFKI (German Research Center for Artificial Intelligence)

3. Job Description
**Brief Summary:**
DFKI is seeking a

/usr/local/lib/python3.10/dist-packages/pydantic/main.py:214: UserWarning: A custom validator is returning a value other than `self`.
Returning anything other than `self` from a top level model validator isn't supported when validating via `__init__`.
See the `model_validator` docs (https://docs.pydantic.dev/latest/concepts/validators/#model-validators) for more details.
  warnings.warn(


 [DEBUG]: == [Job posting information extractor] Task output: {
  "Job_Title": "Researcher - Spatiotemporal Modelling and Event Data Analysis",
  "Company_Name": "DFKI (German Research Center for Artificial Intelligence)",
  "Job_Description": "DFKI is seeking a researcher in Spatiotemporal Modelling and Event Data Analysis to work on projects related to location-based services, mobility data analysis, and event data processing.",
  "Key_Responsibilities": "- Conduct research in Spatiotemporal Modelling and Event Data Analysis\n- Develop algorithms and models for location-based services and mobility data analysis\n- Collaborate with team members and industry partners on research projects\n- Publish research findings in academic journals and present at conferences",
  "Qualifications_and_Requirements": "- Ph.D. or Master's degree in Computer Science, Artificial Intelligence, Geoinformatics, or related fields\n- Strong background in Spatiotemporal Modelling, Data Analysis, and Machine Le

In [ ]:
#test with crewAI sample
from crewai_tools import (
  FileReadTool,
  ScrapeWebsiteTool,
  MDXSearchTool,
  SerperDevTool
)

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()
read_resume = FileReadTool(file_path='/content/drive/MyDrive/CVs/cv.md')
semantic_search_resume = MDXSearchTool(mdx='/content/drive/MyDrive/CVs/cv.md')

profiler = Agent(
    role="Personal Profiler for Engineers",
    goal="Do increditble research on job applicants "
         "to help them stand out in the job market",
    tools = [scrape_tool, search_tool,
             read_resume, semantic_search_resume],
    verbose=True,
    backstory=(
        "Equipped with analytical prowess, you dissect "
        "and synthesize information "
        "from diverse sources to craft comprehensive "
        "personal and professional profiles, laying the "
        "groundwork for personalized resume enhancements."
    )
)

# Task for Profiler Agent: Compile Comprehensive Profile
profile_task = Task(
    description=(
        "Compile a detailed personal and professional profile "
        "using the GitHub ({github_url}) URLs. Utilize tools to extract and "
        "synthesize information from these sources."
    ),
    expected_output=(
        "A comprehensive profile document that includes skills, "
        "project experiences, contributions, interests, and "
        "communication style."
    ),
    agent=profiler,
    async_execution=True
)
job_application_crew = Crew(
    agents=[profiler],

    tasks=[profile_task],

    verbose=True
)
job_application_inputs = {
    'job_posting_url': 'https://jobs.adesso-group.com/AdessoOrange/job/Aachen-Senior-Consultant-SAP-PP-Produktionsplanung-%28all-genders%29-NW-52070/954183155/',
    'github_url': 'https://ymlei.github.io/'
}

In [ ]:
result = job_application_crew.kickoff(inputs=job_application_inputs)

 [DEBUG]: == Working Agent: Personal Profiler for Engineers
 [INFO]: == Starting Task: Compile a detailed personal and professional profile using the GitHub (https://ymlei.github.io/) URLs. Utilize tools to extract and synthesize information from these sources.
 [DEBUG]: == [Personal Profiler for Engineers] Task output: 




> Entering new CrewAgentExecutor chain...
